In [4]:
import openai 
from qdrant_client import QdrantClient
import os

In [4]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [5]:
qdrant_client = QdrantClient(
    url="http://localhost:6333",
)

In [7]:
def preprocess_review(review):
    #  0   main_category   2000 non-null   object 
    #  1   title           2026 non-null   object 
    #  2   average_rating  2026 non-null   float64
    #  3   rating_number   2026 non-null   int64  
    #  4   features        2026 non-null   object 
    #  5   description     2026 non-null   object 
    #  6   price           2026 non-null   float64
    #  7   images          2026 non-null   object 
    #  8   videos          2026 non-null   object 
    #  9   store           2026 non-null   object 
    #  10  categories      2026 non-null   object 
    #  11  details         2026 non-null   object 
    #  12  parent_asin     2026 non-null   object 
    #  13  product_id      2026 non-null   object
    return {
        'product_id': review['product_id'], 
        'main_category': review['main_category'],
        'average_rating': review['average_rating'],
        'rating_number': review['rating_number'],
        'features': review['features'],
        'description': review['description'],
        'price': review['price'],
        'images': review['images'],
        'videos': review['videos'],
        'store': review['store'],
        'categories': review['categories'],
        'details': review['details'],
        'parent_asin': review['parent_asin'],
    }

In [7]:
results = qdrant_client.query_points(
        collection_name="Amazon_Electronics_Data_Collection",
    )

In [10]:
results.points

[ScoredPoint(id=0, version=0, score=1.0, payload={'product_id': 'b86c3534-084b-449b-889e-50bd268ff964', 'text': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET', 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg', 'variant': '

In [9]:
results.points[0].payload

{'product_id': 'b86c3534-084b-449b-889e-50bd268ff964',
 'text': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET',
 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'],
 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': None}],
 'videos': [

In [6]:
def retrieve_data(query, qdrant_client=qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon_Electronics_Data_Collection",
        query=query_embedding,
        limit=k,
        with_payload=True,
    )

    print("results.points: ", results.points)

    retrieved_context_ids = []
    retrieve_context = []
    similarity_scores = []
    retrieved_context_rating_numbers = []
    retrieve_context_main_categories = []
    retrieve_context_prices = []
    retrieve_context_images = []
    retrieve_context_videos = []
    retrieve_context_stores = []
    retrieve_context_categories = []
    retrieve_context_details = []
    retrieve_context_descriptions = []
    retrieve_context_features = []

    for result in results.points:
        payload = result.payload or {}
        retrieved_context_ids.append(result.id)
        similarity_scores.append(result.score)
        retrieve_context.append(payload.get('text', ''))
        retrieved_context_rating_numbers.append(payload.get('rating_number', None))
        retrieve_context_main_categories.append(payload.get('main_category', None))
        retrieve_context_prices.append(payload.get('price', None))
        retrieve_context_images.append(payload.get('images', None))
        retrieve_context_videos.append(payload.get('videos', None))
        retrieve_context_stores.append(payload.get('store', None))
        retrieve_context_categories.append(payload.get('categories', None))
        retrieve_context_details.append(payload.get('details', None))
        retrieve_context_descriptions.append(payload.get('description', None))
        retrieve_context_features.append(payload.get('features', None))

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieve_context': retrieve_context,
        'similarity_scores': similarity_scores,
        'retrieved_context_rating_numbers': retrieved_context_rating_numbers,
        'retrieve_context_main_categories': retrieve_context_main_categories,
        'retrieve_context_prices': retrieve_context_prices,
        'retrieve_context_images': retrieve_context_images,
        'retrieve_context_videos': retrieve_context_videos,
        'retrieve_context_stores': retrieve_context_stores,
        'retrieve_context_categories': retrieve_context_categories,
        'retrieve_context_details': retrieve_context_details,
        'retrieve_context_descriptions': retrieve_context_descriptions,
        'retrieve_context_features': retrieve_context_features,
    }

In [17]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False


In [18]:
retrieve_context = retrieve_data('What are the products type you provide?', k=10)

results.points:  [ScoredPoint(id=496, version=20, score=0.2817377, payload={'product_id': '0513af4b-caa8-4c2b-9e61-31b924e8f657', 'text': 'Baccus Global Llc - V1 Rechargeable Spotlight - Camo Pattern "Product Category: Outdoors/Feeders, Scouting Cameras And Other Hunting Gear"', 'description': [], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41BzB-J7tQL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41BzB-J7tQL._AC_.jpg', 'variant': 'MAIN', 'hi_res': None}], 'videos': [], 'price': 59.99, 'rating_number': 3, 'main_category': 'All Electronics', 'categories': [], 'store': 'Amazon Renewed', 'details': {'Item Weight': '3.2 pounds', 'Item model number': 'MSS1119533Y02', 'Is Discontinued By Manufacturer': 'No', 'Date First Available': 'May 4, 2016'}, 'features': []}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=967, version=24, score=0.26306117, payload={'product_id': '79a359c8-cd78-464a-b671-2d1f3633d569', 'text': 'InstallerParts (10 Pack Etherne

In [19]:
retrieve_context

{'retrieved_context_ids': [496,
  967,
  1780,
  950,
  1484,
  1249,
  1396,
  488,
  147,
  44],
 'retrieve_context': ['Baccus Global Llc - V1 Rechargeable Spotlight - Camo Pattern "Product Category: Outdoors/Feeders, Scouting Cameras And Other Hunting Gear"',
  'InstallerParts (10 Pack Ethernet Cable CAT6 Cable Shielded (SSTP/SFTP) Booted 0.5 FT - Black - Professional Series - 10Gigabit/Sec Network/High Speed Internet Cable, 550MHZ',
  '45W Type C USB C 4X20M26268 Laptop Charger Compatible with Lenovo ThinkPad X1 T480 T580s T580 T480s, Yoga C940 C740 C930 730 S730 730S 920 Chromebook c330 100e 300e 500e c340 N23 Power Supply Cord',
  'C2G USB Extender, Dongle, USB 1.1 over Cat5, SuperBooster Extender, Wall Plate Kit, White, Cables to Go 29342',
  'Fovitec - 1x 12"x12" Photography Display Table - [Gloss Black & Gloss White Finish][Easy Setup][for eCommerce][Magnetic Removable Legs]',
  'Monoprice USB Type C to USB-A 2.0 Cable - 3 Feet - Red, 480Mbps, 2.4A, Braided - Palette Series',


Format retrieved context function

In [1]:
def process_context(context):
    formatted_context = []
    count = len(context.get('retrieved_context_ids', []))

    for index in range(count):
        point_id = context['retrieved_context_ids'][index]
        text = context['retrieve_context'][index] if index < len(context.get('retrieve_context', [])) else ''
        score = context['similarity_scores'][index] if index < len(context.get('similarity_scores', [])) else None
        rating_number = context['retrieved_context_rating_numbers'][index] if index < len(context.get('retrieved_context_rating_numbers', [])) else None
        main_category = context['retrieve_context_main_categories'][index] if index < len(context.get('retrieve_context_main_categories', [])) else ''
        price = context['retrieve_context_prices'][index] if index < len(context.get('retrieve_context_prices', [])) else None
        images = context['retrieve_context_images'][index] if index < len(context.get('retrieve_context_images', [])) else []
        videos = context['retrieve_context_videos'][index] if index < len(context.get('retrieve_context_videos', [])) else []
        store = context['retrieve_context_stores'][index] if index < len(context.get('retrieve_context_stores', [])) else ''
        categories = context['retrieve_context_categories'][index] if index < len(context.get('retrieve_context_categories', [])) else []
        details = context['retrieve_context_details'][index] if index < len(context.get('retrieve_context_details', [])) else ''
        description = context['retrieve_context_descriptions'][index] if index < len(context.get('retrieve_context_descriptions', [])) else ''
        features = context['retrieve_context_features'][index] if index < len(context.get('retrieve_context_features', [])) else []

        formatted_context.append(
            f"ID: {point_id}\n"
            f"Score: {score}\n"
            f"Rating Number: {rating_number}\n"
            f"Main Category: {main_category}\n"
            f"Price: {price}\n"
            f"Store: {store}\n"
            f"Categories: {categories}\n"
            f"Details: {details}\n"
            f"Description: {description}\n"
            f"Features: {features}\n"
            f"Images: {images}\n"
            f"Videos: {videos}\n"
            f"Text: {text}\n---"
        )

    return "\n".join(formatted_context)

In [23]:
preprocessed_context = process_context(retrieve_context)

In [24]:
print(preprocessed_context)

ID: 496
Score: 0.2817377
Title: 
Rating Number: 3
Main Category: All Electronics
Price: 59.99
Store: Amazon Renewed
Categories: []
Details: {'Item Weight': '3.2 pounds', 'Item model number': 'MSS1119533Y02', 'Is Discontinued By Manufacturer': 'No', 'Date First Available': 'May 4, 2016'}
Description: []
Features: []
Images: [{'thumb': 'https://m.media-amazon.com/images/I/41BzB-J7tQL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41BzB-J7tQL._AC_.jpg', 'variant': 'MAIN', 'hi_res': None}]
Videos: []
Model: 
Text: Baccus Global Llc - V1 Rechargeable Spotlight - Camo Pattern "Product Category: Outdoors/Feeders, Scouting Cameras And Other Hunting Gear"
---
ID: 967
Score: 0.26306117
Title: 
Rating Number: 632
Main Category: All Electronics
Price: 22.49
Store: InstallerParts
Categories: ['Electronics', 'Computers & Accessories', 'Computer Accessories & Peripherals', 'Cables & Accessories', 'Cables & Interconnects', 'Ethernet Cables', 'Cat 6 Cables']
Details: {'Brand': 'Installer

Create Prompt Function

In [25]:
def build_prompt(preprocessed_context, question):
    prompt = f"""You are a helpful assistant for answering questions about
      Amazon product reviews. Use the following retrieved context 
      to answer the question. If the context does not contain relevant information, 
      say you don't know.
      Instructions:
       - Use the retrieved context to answer the question.
       - If the context does not contain relevant information, say you don't know.

    Context:
      {preprocessed_context}
      Question: {question}
      Answer:"""
    return prompt

In [26]:
prompt = build_prompt(preprocessed_context, "What kind of Laptop charger do you offer?")
print(prompt)

You are a helpful assistant for answering questions about
      Amazon product reviews. Use the following retrieved context 
      to answer the question. If the context does not contain relevant information, 
      say you don't know.
      Instructions:
       - Use the retrieved context to answer the question.
       - If the context does not contain relevant information, say you don't know.

    Context:
      ID: 496
Score: 0.2817377
Title: 
Rating Number: 3
Main Category: All Electronics
Price: 59.99
Store: Amazon Renewed
Categories: []
Details: {'Item Weight': '3.2 pounds', 'Item model number': 'MSS1119533Y02', 'Is Discontinued By Manufacturer': 'No', 'Date First Available': 'May 4, 2016'}
Description: []
Features: []
Images: [{'thumb': 'https://m.media-amazon.com/images/I/41BzB-J7tQL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41BzB-J7tQL._AC_.jpg', 'variant': 'MAIN', 'hi_res': None}]
Videos: []
Model: 
Text: Baccus Global Llc - V1 Rechargeable Spotlight - Cam

Generate Answer Function

In [27]:
def gen_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [28]:
print(gen_answer(prompt))

We offer a 45W USB-C laptop charger (USB-C Power Delivery). It uses a USB-C port, works with 100-240V AC input, and is compatible with many Lenovo ThinkPad/Yoga models (for example X1, T480/T580s, Yoga C940/C740/C930, 730/S730, 920 Chromebook, and related models).


Combined RAG Pipeline

In [29]:
def Rag_Pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))
    retrieve_context = retrieve_data(question, qdrant_client=qdrant_client, k=top_k)
    preprocessed_context = process_context(retrieve_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = gen_answer(prompt)
    return answer

In [30]:
print(Rag_Pipeline("What kind of Laptop do you offer?", top_k=5))

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_8111/1312403169.py:2: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))


results.points:  [ScoredPoint(id=531, version=21, score=0.45798373, payload={'product_id': '696a0f42-9993-429c-b281-71054dac2a6d', 'text': "Acer Aspire 3 Laptop, 17.3'' HD+ 1600 x 900, 3.60GHz Intel Core i5-1035G, 20GB DDR4 RAM, 128GB SSD + 1TB HDD, DVD-Writer, Windows 10, A317-52", 'description': ["Acer Aspire 3 Laptop, 17.3'' HD+ 1600 x 900, 3.60GHz Intel Core i5-1035G, 20GB DDR4 RAM, 128GB SSD + 1TB HDD, DVD-Writer, Windows 10, A317-52"], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41-Dz1OBLfS._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41-Dz1OBLfS._AC_.jpg', 'variant': 'MAIN', 'hi_res': 'https://m.media-amazon.com/images/I/71o9WGV4E1S._AC_SL1500_.jpg'}, {'thumb': 'https://m.media-amazon.com/images/I/21LRZoGYbSS._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/21LRZoGYbSS._AC_.jpg', 'variant': 'PT01', 'hi_res': 'https://m.media-amazon.com/images/I/61apF7DHVvS._AC_SL1500_.jpg'}, {'thumb': 'https://m.media-amazon.com/images/I/31aEorIsp0S._A